In [ ]:
import pandas as pd
import numpy as np

path = r'/Users/leeseungbo/Desktop/학회 데이터(CSV)/2차 전처리 데이터.csv'
df = pd.read_csv(path)

col = "원인차운전자연령구분"

age_map = {
    "20~30": "20~29",
    "30~40": "30~39",
    "40~50": "40~49",
    "50~60": "50~59",
    "20~29": "20~29",
    "30~39": "30~39",
    "40~49": "40~49",
    "50~59": "50~59",
    "20세 미만": "20세 미만",
    "60세 이상": "60세 이상",
    "기타(잘못입력)": np.nan,
}

# 문자열 정리(양끝 공백 제거)
s = df[col].astype("string").str.strip()

# 매핑 적용
df["연령대_정리"] = s.map(age_map)

# 매핑에서 누락된 값(예: '20~30 ' 같은 변형/오타)이 있는지 점검
unmapped = s[df["연령대_정리"].isna() & s.notna()].unique()
print("매핑 누락 값:", unmapped)

# 분포 확인
print(df["연령대_정리"].value_counts(dropna=False))


In [ ]:
import pandas as pd
import numpy as np

# 0) 데이터 로드
path = '/Users/leeseungbo/Desktop/학회 데이터(CSV)/2차 전처리 데이터.csv'
df = pd.read_csv(path)

age_col = "원인차운전자연령구분"

# 1) 심각도 컬럼 자동 탐색 (파일마다 컬럼명이 달라질 수 있어서)
#    후보 키워드: 심각도/사상/피해/부상/사망 등
severity_candidates = [c for c in df.columns if any(k in c for k in ["심각", "사상", "피해", "부상", "사망", "상해"])]
print("심각도 후보 컬럼:", severity_candidates)

# 여기서 실제 쓰는 심각도 컬럼 하나를 지정하세요.
# 예: severity_col = "사고심각도" 또는 "사상정도" 등
severity_col = severity_candidates[0] if severity_candidates else None
if severity_col is None:
    raise ValueError("심각도 컬럼을 찾지 못했습니다. df.columns에서 심각도 컬럼명을 직접 지정하세요.")

# 2) 연령 문자열 정규화(공백/전각 물결/접미사 제거)
s = df[age_col].astype("string").str.strip()
s = s.str.replace("～", "~", regex=False)      # 전각 물결 통일
s = s.str.replace("세", "", regex=False)      # '세' 제거(있을 경우)
s = s.str.replace(r"\s+", "", regex=True)     # 중간 공백까지 제거(예: '40 ~ 50')

# 3) 연령대 매핑(겹침 제거)
age_map = {
    "20~30": "20~29",
    "30~40": "30~39",
    "40~50": "40~49",
    "50~60": "50~59",
    "20~29": "20~29",
    "30~39": "30~39",
    "40~49": "40~49",
    "50~59": "50~59",
    "20세미만": "20세 미만",
    "20미만": "20세 미만",
    "60세이상": "60세 이상",
    "60이상": "60세 이상",
    "기타(잘못입력)": np.nan,
    "기타": np.nan,
}

df["연령_원본정규화"] = s
df["연령대_정리"] = df["연령_원본정규화"].map(age_map)

# 4) 매핑 누락값 확인
unmapped = df.loc[df["연령대_정리"].isna() & df["연령_원본정규화"].notna(), "연령_원본정규화"].unique()
print("매핑 누락 값(확인 필수):", unmapped)

# 5) 전/후 교차표(Count)
ct_before = pd.crosstab(df[severity_col], df[age_col], dropna=False)
ct_after  = pd.crosstab(df[severity_col], df["연령대_정리"], dropna=False)

print("\n[전] 심각도 x 연령(원본) Count")
print(ct_before)

print("\n[후] 심각도 x 연령대_정리 Count")
print(ct_after)

# 6) 전/후 교차표(행 기준 비율 %)  -> 심각도별 연령 분포 비교 핵심
rowpct_before = ct_before.div(ct_before.sum(axis=1), axis=0) * 100
rowpct_after  = ct_after.div(ct_after.sum(axis=1), axis=0) * 100

print("\n[전] 심각도별 연령 분포(%)")
print(rowpct_before.round(2))

print("\n[후] 심각도별 연령대_정리 분포(%)")
print(rowpct_after.round(2))

# 7) "바뀐 정도" 간단 요약: 후-전(가능한 항목만 비교)
#    원본 구간이 촘촘해서 직접 차이가 안 맞는 경우가 많음. 그래서 여기서는 '후'만 핵심 출력 권장.
print("\n[후] 연령대_정리 전체 분포(Count)")
print(df["연령대_정리"].value_counts(dropna=False))


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

# 0) 데이터 로드
path = r'/Users/leeseungbo/Desktop/학회 데이터(CSV)/2차 전처리 데이터.csv'
df = pd.read_csv(path, low_memory=False)

age_col = "원인차운전자연령구분"

# 1) 연령대 정리(겹침 제거)
s = df[age_col].astype("string").str.strip()
s = s.str.replace("～", "~", regex=False)      # 전각 물결 통일
s = s.str.replace("세", "", regex=False)      # '세' 제거(있으면)
s = s.str.replace(r"\s+", "", regex=True)     # 공백 제거(예: '40 ~ 50')

age_map = {
    "20~30": "20~29",
    "30~40": "30~39",
    "40~50": "40~49",
    "50~60": "50~59",
    "20~29": "20~29",
    "30~39": "30~39",
    "40~49": "40~49",
    "50~59": "50~59",
    "20세미만": "20세 미만",
    "20미만": "20세 미만",
    "60세이상": "60세 이상",
    "60이상": "60세 이상",
    "기타(잘못입력)": np.nan,
    "기타": np.nan,
}

df["연령_원본정규화"] = s
df["연령대_정리"] = df["연령_원본정규화"].map(age_map)

unmapped = df.loc[df["연령대_정리"].isna() & df["연령_원본정규화"].notna(), "연령_원본정규화"].unique()
print("매핑 누락 값(있으면 age_map 보강 필요):", unmapped)

# 2) 심각도 라벨 만들기 (사망/중상/경상)
for c in ["사망", "중상", "경상"]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

conditions = [
    df["사망"] > 0,
    (df["사망"] <= 0) & (df["중상"] > 0),
    (df["사망"] <= 0) & (df["중상"] <= 0) & (df["경상"] > 0),
]
choices = ["사망", "중상", "경상"]
df["심각도"] = np.select(conditions, choices, default="부상없음")

print("\n심각도 분포:")
print(df["심각도"].value_counts(dropna=False))

# 3) 이진화(사망/비사망)
df["fatal"] = (df["사망"] > 0).astype(int)
print("\nfatal 분포:")
print(df["fatal"].value_counts())

# 4) EPDO 지수 생성 (가중치 조절 가능)
W_FATAL = 12
W_SI = 3
W_MI = 1

df["EPDO"] = (W_FATAL * df["사망"]) + (W_SI * df["중상"]) + (W_MI * df["경상"])
print("\nEPDO 요약:")
print(df["EPDO"].describe())

# 5) 심각도 x 연령대(정리 후) 교차표 (Count, Row%)
ct = pd.crosstab(df["심각도"], df["연령대_정리"], dropna=False)
rowpct = ct.div(ct.sum(axis=1), axis=0) * 100

print("\n[후] 심각도 x 연령대_정리 Count")
print(ct)

print("\n[후] 심각도별 연령대 분포(%)")
print(rowpct.round(2))

# 6) 카이제곱 + Cramér’s V (심각도 vs 연령대_정리)
# 분석에서 '부상없음'이 의미 없거나 너무 적으면 제외하는 게 일반적이라 옵션으로 제공
use = df[df["심각도"] != "부상없음"].copy()

tab = pd.crosstab(use["심각도"], use["연령대_정리"])
chi2, p, dof, expected = chi2_contingency(tab)

n = tab.to_numpy().sum()
r, k = tab.shape
cramers_v = np.sqrt((chi2 / n) / (min(r-1, k-1)))

print("\n[카이제곱] 심각도 vs 연령대_정리 (부상없음 제외)")
print("chi2 =", chi2)
print("dof  =", dof)
print("p    =", p)
print("Cramér’s V =", cramers_v)

# 7) (선택) fatal(사망/비사망) x 연령대_정리도 함께 검정
tab2 = pd.crosstab(use["fatal"], use["연령대_정리"])
chi2_2, p2, dof2, _ = chi2_contingency(tab2)
n2 = tab2.to_numpy().sum()
r2, k2 = tab2.shape
cramers_v2 = np.sqrt((chi2_2 / n2) / (min(r2-1, k2-1)))

print("\n[카이제곱] fatal vs 연령대_정리 (부상없음 제외)")
print("chi2 =", chi2_2)
print("dof  =", dof2)
print("p    =", p2)
print("Cramér’s V =", cramers_v2)


In [ ]:
import numpy as np
import pandas as pd

# 1) 부상 발생 여부
df["injury_any"] = ((df["사망"] > 0) | (df["중상"] > 0) | (df["경상"] > 0)).astype(int)

# 2) 부상 있는 데이터만 따로
inj = df[df["injury_any"] == 1].copy()

# 3) 부상 subset에서 3클래스(사망/중상/경상)
conditions = [
    inj["사망"] > 0,
    (inj["사망"] <= 0) & (inj["중상"] > 0),
    (inj["사망"] <= 0) & (inj["중상"] <= 0) & (inj["경상"] > 0),
]
choices = ["사망", "중상", "경상"]

inj["sev3"] = np.select(conditions, choices, default=None)  # None = object dtype
inj["sev3"] = inj["sev3"].astype("string")                  # (선택) 문자열 dtype로 통일
inj.loc[inj["sev3"].isna(), "sev3"] = pd.NA                 # (선택) 결측 표준화


print("injury_any 분포:\n", df["injury_any"].value_counts())
print("\n부상 subset sev3 분포:\n", inj["sev3"].value_counts(dropna=False))

In [ ]:
cols = ["차량상태(언더라이드)", "차량상태(잭나이프)"]
for c in cols:
    vc = df[c].value_counts(dropna=False)
    print("\n", c)
    print(vc)
    print("1 비율 =", (df[c]==1).mean())

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# 1) injury_any 생성 (없으면 생성)
if "injury_any" not in df.columns:
    for c in ["사망","중상","경상"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)
    df["injury_any"] = ((df["사망"]>0) | (df["중상"]>0) | (df["경상"]>0)).astype(int)

# 2) sev3 생성 (부상 있는 행에 대해서만)
df["sev3"] = pd.NA
mask = df["injury_any"] == 1
df.loc[mask & (df["경상"]>0), "sev3"] = "경상"
df.loc[mask & (df["중상"]>0), "sev3"] = "중상"
df.loc[mask & (df["사망"]>0), "sev3"] = "사망"  # 우선순위 highest
df["sev3"] = df["sev3"].astype("string")

# 3) 이제 Fisher 분석 진행용 subset
sev = df[df["injury_any"] == 1].copy()

# 타깃 이진화
sev["fatal_bin"] = (sev["sev3"] == "사망").astype(int)
sev["severe_bin"] = sev["sev3"].isin(["중상", "사망"]).astype(int)

def or_ci_fisher(tab_2x2):
    tab = tab_2x2.astype(float)
    if (tab == 0).any():
        tab = tab + 0.5
    a, b = tab[0,0], tab[0,1]
    c, d = tab[1,0], tab[1,1]
    OR = (d * a) / (b * c)
    se = np.sqrt(1/a + 1/b + 1/c + 1/d)
    lo = np.exp(np.log(OR) - 1.96*se)
    hi = np.exp(np.log(OR) + 1.96*se)
    return OR, lo, hi

results = []
for col in ["차량상태(언더라이드)", "차량상태(잭나이프)"]:
    sev[col+"_bin"] = sev[col].isin([1,2]).astype(int)

    for target, tname in [("fatal_bin", "사망 vs 비사망"),
                          ("severe_bin", "중상+사망 vs 경상")]:
        tab = pd.crosstab(sev[target], sev[col+"_bin"]).reindex(index=[0,1], columns=[0,1], fill_value=0)

        OR_f, p = fisher_exact(tab.values)
        OR, lo, hi = or_ci_fisher(tab.values)

        results.append({
            "변수": col,
            "비교": tname,
            "노출(발생)건수": int(sev[col+"_bin"].sum()),
            "Fisher_p": p,
            "OR": OR,
            "OR_95%CI": (lo, hi),
            "2x2표": tab.values.tolist(),
        })

res = pd.DataFrame(results)
print(res[["변수","비교","노출(발생)건수","Fisher_p","OR","OR_95%CI","2x2표"]])


In [ ]:
cols = ["사고후상태(전도)", "사고후상태(전복)"]
for c in cols:
    vc = df[c].value_counts(dropna=False)
    print("\n", c)
    print(vc)
    print("1 비율 =", (df[c]==1).mean())

### XGBoost 분석 

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

import xgboost as xgb

# =========================
# 1) Load
# =========================
path = r'/Users/leeseungbo/Desktop/학회 데이터(CSV)/2차 전처리 데이터.csv'
df = pd.read_csv(path, low_memory=False)

# =========================
# 2) Targets 생성: injury_any, sev3
# =========================
for c in ["사망", "중상", "경상"]:
    if c not in df.columns:
        raise KeyError(f"필수 컬럼 누락: {c}")
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["injury_any"] = ((df["사망"]>0) | (df["중상"]>0) | (df["경상"]>0)).astype(int)

df["sev3"] = pd.NA
mask = df["injury_any"] == 1
df.loc[mask & (df["경상"]>0), "sev3"] = "경상"
df.loc[mask & (df["중상"]>0), "sev3"] = "중상"
df.loc[mask & (df["사망"]>0), "sev3"] = "사망"   # 우선순위 highest
df["sev3"] = df["sev3"].astype("string")

print("injury_any 분포:\n", df["injury_any"].value_counts())
print("\nsev3 분포(부상 subset):\n", df.loc[df["injury_any"]==1, "sev3"].value_counts(dropna=False))

# =========================
# 3) 언더라이드/잭나이프 제외
# =========================
drop_rare = ["차량상태(언더라이드)", "차량상태(잭나이프)",
             "차량상태(언더라이드)_bin", "차량상태(잭나이프)_bin"]
drop_rare = [c for c in drop_rare if c in df.columns]
df = df.drop(columns=drop_rare, errors="ignore")
print("\n제외 컬럼:", drop_rare)

# =========================
# 4) Feature matrix 구성 (누수 변수 제거)
# =========================
leak_cols = ["사망","중상","경상","심각도","fatal","EPDO","sev3","injury_any"]
drop_cols = [c for c in leak_cols if c in df.columns]

X = df.drop(columns=drop_cols, errors="ignore").copy()

# 고유값 과다(사고ID류) 자동 제거(옵션)
high_card_cols = []
for c in X.columns:
    if X[c].dtype == "object" or str(X[c].dtype).startswith("string"):
        if X[c].nunique(dropna=True) > 0.5 * len(X):
            high_card_cols.append(c)

X = X.drop(columns=high_card_cols, errors="ignore")
print("고유값 과다 제거 컬럼 수:", len(high_card_cols))

# 원-핫 + 결측 처리
X = pd.get_dummies(X, dummy_na=True)
X = X.replace([np.inf, -np.inf], np.nan).fillna(-999)

# =========================
# 5) Stage 1: injury_any (부상여부) - 불균형 대응
# =========================
y1 = df["injury_any"].astype(int)

X1_train, X1_test, y1_train, y1_test = train_test_split(
    X, y1, test_size=0.2, random_state=42, stratify=y1
)

neg, pos = (y1_train==0).sum(), (y1_train==1).sum()
spw = neg / pos
print("\n[Stage1] scale_pos_weight:", spw)

clf1 = xgb.XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    min_child_weight=1,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=spw,
    tree_method="hist",
    random_state=42,
)

clf1.fit(X1_train, y1_train)

p1 = clf1.predict_proba(X1_test)[:, 1]
print("\n[Stage1] AUC    :", round(roc_auc_score(y1_test, p1), 4))
print("[Stage1] PR-AUC :", round(average_precision_score(y1_test, p1), 4))

# =========================
# 6) Stage 2: 부상 subset sev3 (경상/중상/사망) - 멀티클래스
# =========================
inj = df[df["injury_any"]==1].copy()
X2 = X.loc[inj.index].copy()

label_order = ["경상", "중상", "사망"]
label_map = {k:i for i,k in enumerate(label_order)}
y2 = inj["sev3"].map(label_map).astype(int)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

# 클래스 가중치(빈도 역수)
class_counts = y2_train.value_counts().sort_index()
class_weights = (class_counts.sum() / (len(class_counts) * class_counts)).to_dict()
w_train = y2_train.map(class_weights).values
print("\n[Stage2] class_counts:\n", class_counts)
print("[Stage2] class_weights:\n", class_weights)

clf2 = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42,
)

clf2.fit(X2_train, y2_train, sample_weight=w_train)

pred2 = clf2.predict(X2_test)

print("\n[Stage2] classification report (macro avg 중요):")
print(classification_report(y2_test, pred2, target_names=label_order, digits=4))

print("[Stage2] confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y2_test, pred2))


In [ ]:
imp = pd.Series(clf1.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(30))

In [ ]:
# 기존 class_weights 만든 뒤
class_weights[1] *= 1.5   # 또는 2.0까지 시도
w_train = y2_train.map(class_weights).values


In [ ]:
from sklearn.metrics import f1_score, balanced_accuracy_score

print("Stage2 macro-F1:", f1_score(y2_test, pred2, average="macro"))
print("Stage2 weighted-F1:", f1_score(y2_test, pred2, average="weighted"))
print("Stage2 balanced acc:", balanced_accuracy_score(y2_test, pred2))

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

# y2: 0=경상, 1=중상, 2=사망 (label_order = ["경상","중상","사망"])
label_order = ["경상", "중상", "사망"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

# 기본 클래스 가중치(빈도 역수)
class_counts = y2_train.value_counts().sort_index()
base_weights = (class_counts.sum() / (len(class_counts) * class_counts)).to_dict()

def train_multiclass(extra_mid_weight=1.0):
    cw = base_weights.copy()
    cw[1] = cw[1] * extra_mid_weight  # 중상(1)만 증폭
    w_train = y2_train.map(cw).values

    clf = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        objective="multi:softprob",
        num_class=3,
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
    )
    clf.fit(X2_train, y2_train, sample_weight=w_train)

    pred = clf.predict(X2_test)
    macro = f1_score(y2_test, pred, average="macro")
    balacc = balanced_accuracy_score(y2_test, pred)

    print(f"\n=== extra_mid_weight={extra_mid_weight} ===")
    print("macro-F1:", round(macro, 4), "| balanced acc:", round(balacc, 4))
    print(classification_report(y2_test, pred, target_names=label_order, digits=4))
    print("confusion matrix:\n", confusion_matrix(y2_test, pred))

    return clf, pred

# 1.0(기본), 1.5, 2.0 비교
clf2_w1, _ = train_multiclass(1.0)
clf2_w15, _ = train_multiclass(1.5)
clf2_w2, _ = train_multiclass(2.0)


In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score

label_order = ["경상", "중상", "사망"]

# y2: 0=경상, 1=중상, 2=사망
# --------------------------
# Step 1: 경상(0) vs 중증(1)
# --------------------------
y_severe = (y2 != 0).astype(int)   # 중상/사망이면 1

X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(
    X2, y_severe, test_size=0.2, random_state=42, stratify=y_severe
)

neg, pos = (y_a_train==0).sum(), (y_a_train==1).sum()
spw_a = neg / pos

clf_a = xgb.XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=spw_a,
    tree_method="hist",
    random_state=42,
)
clf_a.fit(X_a_train, y_a_train)

p_a = clf_a.predict_proba(X_a_test)[:,1]
pred_a = (p_a >= 0.5).astype(int)  # 임계값은 필요하면 튜닝

print("\n[Hier-1] 경상 vs 중증")
print(confusion_matrix(y_a_test, pred_a))
print(classification_report(y_a_test, pred_a, digits=4))


# ----------------------------------------
# Step 2: 중증 subset에서 중상(0) vs 사망(1)
# ----------------------------------------
# 학습/평가는 '중증(=y_severe=1)'인 표본만 사용
idx_severe = (y_severe == 1)
X_sev = X2.loc[idx_severe].copy()
y_sev = y2.loc[idx_severe].copy()

# 중증 내에서: 중상(1)->0, 사망(2)->1
y_fatal = (y_sev == 2).astype(int)

X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
    X_sev, y_fatal, test_size=0.2, random_state=42, stratify=y_fatal
)

neg, pos = (y_b_train==0).sum(), (y_b_train==1).sum()
spw_b = neg / pos

clf_b = xgb.XGBClassifier(
    n_estimators=800,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary:logistic",
    eval_metric="aucpr",
    scale_pos_weight=spw_b,
    tree_method="hist",
    random_state=42,
)
clf_b.fit(X_b_train, y_b_train)

p_b = clf_b.predict_proba(X_b_test)[:,1]
pred_b = (p_b >= 0.5).astype(int)

print("\n[Hier-2] 중상 vs 사망 (중증 subset)")
print(confusion_matrix(y_b_test, pred_b))
print(classification_report(y_b_test, pred_b, digits=4))


# ----------------------------------------
# Step 3: 계층형으로 최종 3클래스 예측 생성(테스트셋 기준)
# ----------------------------------------
# 계층형 평가는 같은 테스트셋에서 해야 하므로,
# 아래는 "X2_test"를 기준으로 3클래스 예측을 만들고 macro-F1 계산하는 간단 버전임.
# (엄밀히는 동일 split으로 계층 학습/예측 파이프라인을 맞추는 게 이상적이지만,
#  우선 성능 감 잡는 용도)

# 3클래스 예측: 기본은 경상(0)
pred3 = np.zeros(len(X2_test), dtype=int)

# 1단계 중증 예측
p1 = clf_a.predict_proba(X2_test)[:,1]
is_severe = (p1 >= 0.5)

# 중증으로 판단된 것들만 2단계(사망 vs 중상)
if is_severe.any():
    p2 = clf_b.predict_proba(X2_test[is_severe])[:,1]
    is_fatal = (p2 >= 0.5)

    # 중증이면 기본 중상(1)
    pred3[is_severe] = 1
    # 그 중 사망이면 2
    pred3[np.where(is_severe)[0][is_fatal]] = 2

macro = f1_score(y2_test, pred3, average="macro")
balacc = balanced_accuracy_score(y2_test, pred3)

print("\n[Hier Final] 3-class 성능")
print("macro-F1:", round(macro, 4), "| balanced acc:", round(balacc, 4))
print(classification_report(y2_test, pred3, target_names=label_order, digits=4))
print("confusion matrix:\n", confusion_matrix(y2_test, pred3))


In [ ]:
for th in [0.35, 0.4, 0.45, 0.5]:
    p1 = clf_a.predict_proba(X2_test)[:,1]
    is_severe = (p1 >= th)

    pred3 = np.zeros(len(X2_test), dtype=int)
    if is_severe.any():
        p2 = clf_b.predict_proba(X2_test[is_severe])[:,1]
        is_fatal = (p2 >= 0.5)
        pred3[is_severe] = 1
        pred3[np.where(is_severe)[0][is_fatal]] = 2

    macro = f1_score(y2_test, pred3, average="macro")
    balacc = balanced_accuracy_score(y2_test, pred3)
    print("th=", th, "macro-F1=", round(macro,4), "balacc=", round(balacc,4))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, balanced_accuracy_score
import numpy as np
import pandas as pd

label_order = ["경상","중상","사망"]

def predict_hier(X_input, th1=0.5, th2=0.5):
    p1 = clf_a.predict_proba(X_input)[:, 1]
    is_severe = (p1 >= th1)

    pred3 = np.zeros(len(X_input), dtype=int)  # 0=경상
    if is_severe.any():
        p2 = clf_b.predict_proba(X_input[is_severe])[:, 1]
        is_fatal = (p2 >= th2)

        pred3[is_severe] = 1  # 중증이면 기본 중상
        pred3[np.where(is_severe)[0][is_fatal]] = 2  # 사망
    return pred3

# 최종 예측
pred3 = predict_hier(X2_test, th1=0.5, th2=0.5)

# 성능
macro = f1_score(y2_test, pred3, average="macro")
balacc = balanced_accuracy_score(y2_test, pred3)
cm = confusion_matrix(y2_test, pred3)

print("=== Hier Final (th1=0.5, th2=0.5) ===")
print("macro-F1:", round(macro, 4), "| balanced acc:", round(balacc, 4))
print("confusion matrix (rows=true, cols=pred):\n", cm)
print(classification_report(y2_test, pred3, target_names=label_order, digits=4))

# (선택) 결과 df로 저장
final_report = pd.DataFrame(
    classification_report(y2_test, pred3, target_names=label_order, digits=4, output_dict=True)
).T
final_report["macro_F1"] = macro
final_report["balanced_acc"] = balacc
final_report


In [ ]:
import re

# 1) 원본 df 기준으로 누수 후보 컬럼 찾기
leak_patterns = [
    r"사망",           # '사망' 포함 컬럼 전부 1차 후보 (너무 공격적이면 예외처리)
    r"중상",
    r"경상",
    r"피해",           # 차량피해/인명피해 등
    r"사상자",
    r"부상",
    r"EPDO",
    r"sev3",
    r"injury_any",
]

# 예외로 살리고 싶은 컬럼 있으면 여기에 넣기(예: 사고등급이 사전정보라면 남길 수 있음)
allowlist = set([
    # 예: "사고등급",  # 이게 사전 분류 기준이면 가능. 다만 대개 결과성 변수임 → 보통은 제외 추천
])

leak_cols = []
for c in df.columns:
    if c in allowlist:
        continue
    for pat in leak_patterns:
        if re.search(pat, c):
            leak_cols.append(c)
            break

leak_cols = sorted(set(leak_cols))
print("누수 의심 컬럼 수:", len(leak_cols))
print(leak_cols[:50])


In [ ]:
import shap

# 샘플링(속도)
X_shap1 = X2_test.sample(n=min(2000, len(X2_test)), random_state=42)

explainer_a = shap.TreeExplainer(clf_a)
sv_a = explainer_a.shap_values(X_shap1)

# 요약 플롯
shap.summary_plot(sv_a, X_shap1, show=True)

# Top 20 변수(절대 SHAP 평균)
imp_a = pd.Series(np.abs(sv_a).mean(axis=0), index=X_shap1.columns).sort_values(ascending=False)
print("\n[Hier-1] Top20 features:")
print(imp_a.head(20))


In [ ]:
# 중증 subset만 SHAP
idx_severe_test = ( (clf_a.predict_proba(X2_test)[:,1]) >= 0.5 )
X_severe_test = X2_test[idx_severe_test].copy()

# 너무 작으면 샘플링 최소 보정
X_shap2 = X_severe_test.sample(n=min(2000, len(X_severe_test)), random_state=42)

explainer_b = shap.TreeExplainer(clf_b)
sv_b = explainer_b.shap_values(X_shap2)

shap.summary_plot(sv_b, X_shap2, show=True)

imp_b = pd.Series(np.abs(sv_b).mean(axis=0), index=X_shap2.columns).sort_values(ascending=False)
print("\n[Hier-2] Top20 features:")
print(imp_b.head(20))

#### 2차 작업

In [ ]:
import numpy as np
import pandas as pd

path = r'/Users/leeseungbo/Desktop/학회 데이터(CSV)/2차 전처리 데이터.csv'
df = pd.read_csv(path, low_memory=False)

# =========================
# A) (선택) 타깃/누수 컬럼은 일단 제외하고 X만 만들기
#    공선성 정리는 '피처끼리'라서 타깃은 빼고 작업하는 게 정석
# =========================
leak_cols = ["사망","중상","경상","심각도","fatal","EPDO","sev3","injury_any"]
X0 = df.drop(columns=[c for c in leak_cols if c in df.columns], errors="ignore").copy()

# =========================
# B) 원-핫(범주형) + 결측 더미
# =========================
X = pd.get_dummies(X0, dummy_na=True)

# 숫자화/무한대/결측 처리
X = X.replace([np.inf, -np.inf], np.nan)

# =========================
# 1) 상수/준상수(거의 변하지 않는 컬럼) 제거
# =========================
# 상수
const_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]

# 준상수(한 값이 99.9% 이상) -> 필요하면 0.999를 조정
near_const_cols = []
thr = 0.999
for c in X.columns:
    vc = X[c].value_counts(dropna=False, normalize=True)
    if len(vc) > 0 and vc.iloc[0] >= thr:
        near_const_cols.append(c)

drop1 = sorted(set(const_cols + near_const_cols))
X1 = X.drop(columns=drop1, errors="ignore")
print("상수/준상수 제거:", len(drop1))

# =========================
# 2) 완전 중복/완전 상관(상관=1.0) 제거
#    (원-핫에서 드물지만 종종 생김)
# =========================
# float 변환(상관계산 위해)
X1f = X1.astype(float)

corr = X1f.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

perfect_drop = [col for col in upper.columns if any(upper[col] >= 0.999999)]
X2 = X1.drop(columns=perfect_drop, errors="ignore")
print("완전(거의)상관 제거:", len(perfect_drop))

# =========================
# 3) 강상관 클러스터에서 대표변수만 남기기 (|r|>=0.95)
#    - 같은 '상관 그룹'에서 하나만 남기고 나머지 제거
#    - 대표 선택 기준: 결측 적고(=정보 많고), 분산 큰 변수 우선
# =========================
X2f = X2.astype(float)
corr2 = X2f.corr().abs()
threshold = 0.95

to_drop = set()
cols = corr2.columns.tolist()
kept = []

# 간단한 greedy 방식(빠르고 실무에서 충분히 잘 동작)
for i, c in enumerate(cols):
    if c in to_drop:
        continue
    # c와 강상관인 컬럼들
    group = corr2.index[corr2[c] >= threshold].tolist()

    # 아직 드롭되지 않은 멤버만
    group = [g for g in group if g not in to_drop]

    # 그룹이 1개면 그냥 유지
    if len(group) == 1:
        kept.append(c)
        continue

    # 대표 선택: 결측 적음 + 분산 큼
    scores = {}
    for g in group:
        miss = X2[g].isna().mean()
        var = X2[g].var(skipna=True)
        scores[g] = (-miss, var)  # miss 적을수록, var 클수록 좋음

    rep = sorted(scores, key=lambda k: scores[k], reverse=True)[0]
    kept.append(rep)

    for g in group:
        if g != rep:
            to_drop.add(g)

X3 = X2.drop(columns=sorted(to_drop), errors="ignore")
print("강상관(|r|>=0.95) 제거:", len(to_drop))
print("최종 피처 수:", X3.shape[1])


In [ ]:
import pandas as pd
import numpy as np

date_col = "사고일자"      # 네 컬럼명으로 수정
time_col = "사고시간"      # 네 컬럼명으로 수정 (예: 1320, "13:20", "1320" 등)

# 1) 날짜 파싱
df[date_col] = pd.to_datetime(df[date_col], errors="coerce")

df["weekday"] = df[date_col].dt.weekday          # 0=월 ... 6=일
df["is_weekend"] = df["weekday"].isin([5,6]).astype(int)
df["month"] = df[date_col].dt.month
df["year"] = df[date_col].dt.year  # 연도 효과를 보고 싶을 때만(보통은 제외/더미 처리)

# 계절
def month_to_season(m):
    if pd.isna(m): return np.nan
    m = int(m)
    if m in [3,4,5]: return "봄"
    if m in [6,7,8]: return "여름"
    if m in [9,10,11]: return "가을"
    return "겨울"
df["season"] = df["month"].apply(month_to_season)

# 2) 시간 파싱 (케이스 대응)
# - "13:20" 형태면 그대로 파싱
# - 1320 / "1320" 형태면 HHMM으로 해석
t = df[time_col].astype("string").str.strip()

# "HH:MM"이면 그대로
mask_colon = t.str.contains(":", na=False)
hour = pd.Series(np.nan, index=df.index, dtype="float")

hour.loc[mask_colon] = pd.to_datetime(t.loc[mask_colon], format="%H:%M", errors="coerce").dt.hour

# 그 외는 HHMM(예: 1320, 0450)
t2 = t.loc[~mask_colon].str.zfill(4)
hh = pd.to_numeric(t2.str[:2], errors="coerce")
mm = pd.to_numeric(t2.str[2:], errors="coerce")
hour.loc[~mask_colon] = hh.where((hh>=0)&(hh<=23))

df["hour"] = hour

# 3) 시간대 bin
bins = [-1, 5, 9, 15, 19, 23]
labels = ["심야(0-5)","출근(6-9)","주간(10-15)","퇴근(16-19)","야간(20-23)"]
df["time_bin"] = pd.cut(df["hour"], bins=bins, labels=labels)

# 4) 주기성 sin/cos
df["hour_sin"] = np.sin(2*np.pi*df["hour"]/24)
df["hour_cos"] = np.cos(2*np.pi*df["hour"]/24)
df["month_sin"] = np.sin(2*np.pi*df["month"]/12)
df["month_cos"] = np.cos(2*np.pi*df["month"]/12)

# 5) 원본 컬럼은 제거(권장)
df = df.drop(columns=[date_col, time_col], errors="ignore")


In [ ]:
import numpy as np
import pandas as pd

# 타깃/누수 후보는 공선성 계산에서 제외(있으면)
leak_cols = ["사망","중상","경상","심각도","fatal","EPDO","sev3","injury_any"]
X0 = df.drop(columns=[c for c in leak_cols if c in df.columns], errors="ignore").copy()

X = pd.get_dummies(X0, dummy_na=True).replace([np.inf,-np.inf], np.nan)

# 1) 상수/준상수 제거
const_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
near_const_cols = []
thr = 0.999
for c in X.columns:
    vc = X[c].value_counts(dropna=False, normalize=True)
    if len(vc) > 0 and vc.iloc[0] >= thr:
        near_const_cols.append(c)

drop1 = sorted(set(const_cols + near_const_cols))
X1 = X.drop(columns=drop1, errors="ignore")
print("상수/준상수 제거:", len(drop1))

# 2) 완전(거의)상관 제거
X1f = X1.fillna(0).astype(float)
corr = X1f.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
perfect_drop = [col for col in upper.columns if any(upper[col] >= 0.999999)]
X2 = X1.drop(columns=perfect_drop, errors="ignore")
print("완전상관 제거:", len(perfect_drop))

# 3) 강상관(|r|>=0.95) 제거(대표만 남김)
X2f = X2.fillna(0).astype(float)
corr2 = X2f.corr().abs()
threshold = 0.95

to_drop = set()
cols = corr2.columns.tolist()

for c in cols:
    if c in to_drop:
        continue
    group = corr2.index[corr2[c] >= threshold].tolist()
    group = [g for g in group if g not in to_drop]
    if len(group) <= 1:
        continue

    # 대표 선택: 결측 적고 분산 큰 것
    scores = {}
    for g in group:
        miss = X2[g].isna().mean()
        var = X2[g].var(skipna=True)
        scores[g] = (-miss, var)

    rep = sorted(scores, key=lambda k: scores[k], reverse=True)[0]
    for g in group:
        if g != rep:
            to_drop.add(g)

X3 = X2.drop(columns=sorted(to_drop), errors="ignore")
print("강상관 제거:", len(to_drop))
print("최종 피처 수:", X3.shape[1])


In [ ]:
col = "인명구조소요시간"  # 확인할 컬럼명
print(df[col].dropna().unique())

In [ ]:
df['인명구조소요시간'].unique()